# 04 — Noise, significance, and how long you must wait

*Weeks 5–6. This is the heart of the project.*

This notebook distinguishes two quantities: **expected significance**, which defines the smooth headline sensitivity curve, and **observed significance**, which fluctuates because a simulated measurement is a Poisson random draw.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../code'))
import numpy as np
import matplotlib.pyplot as plt
import muography as mg
import project_config as cfg
import student_analysis as sa
print('ready')


## Poisson counting

If the expected number of events is N, the typical statistical spread is about √N. Use the fixed reference seed while developing so that your noisy figures are reproducible.


In [ ]:
rng = np.random.default_rng(cfg.REFERENCE_SEED)
for mean in [10, 100, 1000, 10000]:
    draws = rng.poisson(mean, size=5000)
    print(f'expect {mean:6d}   actual spread = {draws.std():8.1f}   '
          f'(sqrt of mean = {np.sqrt(mean):8.1f})   relative = {draws.std()/mean:6.2%}')


## Build the baseline expectation maps

This notebook deliberately rebuilds its inputs from the shared functions. You can run it from a fresh kernel without running notebook 03 first.


In [ ]:
target = mg.Pyramid(
    base=cfg.PYRAMID_BASE_M, height=cfg.PYRAMID_HEIGHT_M,
    density=cfg.ROCK_DENSITY_G_CM3, void_centre=cfg.VOID_CENTRE_M,
    void_radius=cfg.VOID_RADIUS_M, has_void=True,
)
target0 = target.without_void()
detector = np.array(cfg.DETECTOR_POSITION_M, dtype=float)
expected_1d, ax_deg, ay_deg = sa.expected_counts(target, detector, 1.0)
expected_1d_0, _, _ = sa.expected_counts(target0, detector, 1.0)
expected_30d, _, _ = sa.expected_counts(target, detector, 30.0)
expected_30d_0, _, _ = sa.expected_counts(target0, detector, 30.0)


### YOUR TURN — make the images noisy

Use `mg.poisson_counts` with the reference seed to make 1-day and 30-day observed count maps. Plot each observed map divided by the corresponding **expected no-void** map.


In [ ]:
rng = np.random.default_rng(cfg.REFERENCE_SEED)
observed_1d = mg.poisson_counts(expected_1d, rng)
observed_30d = mg.poisson_counts(expected_30d, rng)


## Observed significance

`mg.significance(observed, expected_no_void)` compares one noisy observation with the no-chamber expectation. It fluctuates from run to run. For this educational project, 5σ is a predefined headline threshold; it is not a complete discovery test for a real experiment.

### YOUR TURN

Make significance maps for 1 day and 30 days. Record the random seed in your figure caption or log. Do not define the official exposure time from the largest noisy pixel.


In [ ]:
S1_obs = mg.significance(observed_1d, expected_1d_0)
S30_obs = mg.significance(observed_30d, expected_30d_0)


## The main result: expected significance

The official headline curve uses **expected significance**:

`S_expected = (expected_with_void - expected_without_void) / sqrt(expected_without_void)`

This is deterministic for fixed parameters and follows the expected `√T` scaling. It is the correct quantity for a reproducible exposure-time estimate in this simplified project.


In [ ]:
days_list = np.array([1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60, 90, 120, 180], dtype=float)
# Choose the signal pixel once from the baseline expected signal.
base_mu_v, _, _ = sa.expected_counts(target, detector, 1.0)
base_mu_0, _, _ = sa.expected_counts(target0, detector, 1.0)
signal_idx = np.unravel_index(np.argmax(base_mu_v - base_mu_0), base_mu_v.shape)

S_expected = []
for days in days_list:
    mu_v, _, _ = sa.expected_counts(target, detector, days)
    mu_0, _, _ = sa.expected_counts(target0, detector, days)
    S_map = mg.expected_significance(mu_v, mu_0)
    S_expected.append(S_map[signal_idx])
S_expected = np.asarray(S_expected)
print('signal pixel:', signal_idx)
for d, s in zip(days_list, S_expected):
    print(f'{d:6.0f} days  expected S = {s:6.3f}')


### YOUR TURN — plot the exposure curve

Plot `S_expected` against days on a logarithmic x-axis. Add horizontal lines at 3σ and 5σ. Determine the first crossing of 5σ by interpolation between the nearest scan points.

Your headline sentence should be:

> **For the baseline 0.25 m² detector and 10 m radius chamber, the expected signal reaches 5σ after approximately ___ days under the project's simplified model.**

Do not add an uncertainty to this number unless your mentor gives you a defined procedure for estimating one.


In [ ]:
# YOUR TURN


## Why the noisy curve is different

If you repeat the analysis with Poisson observations, individual significance values will jitter around the expected curve. That is not a bug. It is the statistical fluctuation the simulation is designed to teach.

The previous version of this project used the maximum noisy pixel as the headline statistic. We no longer do that: choosing a pixel because it fluctuated upward introduces a selection bias. You may explore the maximum as a separate exercise, but it is not the official exposure-time definition.


## Optional: the no-void control

Run the same noisy significance calculation on `target.without_void()`. This gives you a concrete demonstration that a noise-only image can contain locally surprising pixels. In your report, call this a **control** or **null simulation**, not a discovery test.


In [ ]:
# YOUR TURN (optional)
